# Neural machine translation (Aplications of Natural Language Processing)

In this colab notebook you will learn how to work with a pre-trained model from [Huggingface](https://huggingface.co/) and how to fine tune such a model to improve its performace on your texts.

----
The code in this colab notebook is insprired in the content of the following websites:
* https://medium.com/@tskumar1320/how-to-fine-tune-pre-trained-language-translation-model-3e8a6aace9f
* https://huggingface.co/docs/transformers/training
* https://huggingface.co/docs/transformers/main_classes/tokenizer
* https://huggingface.co/docs/evaluate/


## Install the required libraries

In [ ]:
! pip install transformers[torch,sentencepiece]
! pip install datasets
! pip install evaluate
! pip install sacremoses
! pip install sacrebleu

## Give access to your Google Drive and set some variables

List of variables to set:
* `mydrive`: Full path to the folder in Google Drive where the corpora to be used for fine tuning is located
* `source`: Source language (see [ISO-639-1 two-letter codes](https://en.wikipedia.org/wiki/List_of_ISO_639-1_codes))
* `target`: Target language (see [ISO-639-1 two-letter codes](https://en.wikipedia.org/wiki/List_of_ISO_639-1_codes))
* `corpus`: Prefix of the files with the parallel corpus in moses format (two documents with the same number of lines; no blank line in either document, no duplicated parallel entries)
* `model_name`: Name of the pre-trained MT model to be used. You can use [any of the models](https://huggingface.co/Helsinki-NLP) made available by the NLP Language Technology Research Group at the University of Helsinki. Other models could also be used.
* `output_model_name`: Folder where the model after fine tuning will be saved.
* `train_size`: Amount of parallel sentences to be used for training (fine tuning).
* `test_size`: Amount of parallel sentences to be used for testing.
* `dev_size`: Amount of parallel sentences to be used for development.
* `patience`: Patience to be used for early stoppping.
* `batch_size`: Number of training samples to be used in each training step.

Note that the parallel corpus to be used for fine tuning must consist of at least `train_size`+`test_size`+`dev_size` parallel sentences.

In [ ]:
## 6. Corpus preprocessing and cleaning

Before constructing the final datasets used in the experiments, the downloaded MultiUN bilingual corpus is preprocessed using the script `preprocess-corpus.sh` provided with the assignment. This step is necessary because all later corpus splits must be derived from a cleaned and structurally consistent pool of parallel sentence pairs.

The purpose of this preprocessing stage is to improve corpus quality before any training, development, test, or monolingual subsets are created. In particular, the script is used to remove duplicated parallel segments, eliminate sentence pairs in which one side is empty, normalize spacing, and preserve the alignment between the English and Spanish files. Since the corpus is handled in Moses format, maintaining exact line-by-line correspondence between both sides is essential.

Preprocessing is performed before any dataset extraction takes place. This means that the bilingual training set, the development set, the test set, and the material later used to derive the monolingual subsets are all obtained from the cleaned output of this stage rather than directly from the raw downloaded corpus. This ordering is important because it ensures that all later subsets are based on consistent and validated input data.

After preprocessing, several quality conditions must hold. The English and Spanish files must contain the same number of lines, no blank lines should remain, duplicate parallel pairs should have been removed, and sentence alignment must remain consistent across both sides of the corpus. These checks are essential because the reliability of the entire experiment depends on the quality of this cleaned bilingual pool.

For this reason, corpus preprocessing is not a minor preparatory step but a necessary foundation for the later stages of splitting, training, synthetic data generation, and BLEU-based evaluation in the English↔Spanish iterative back-translation workflow.

## SECTION 6

In [7]:
# =========================
# CELL 0: DOWNLOAD + UNZIP NEWS-COMMENTARY EN-ES
# =========================

from google.colab import drive
from pathlib import Path
import subprocess

drive.mount("/content/drive", force_remount=True)

mydrive = Path("/content/drive/MyDrive/anlp")
mydrive.mkdir(parents=True, exist_ok=True)

zip_path = mydrive / "en-es.txt.zip"
download_url = "https://object.pouta.csc.fi/OPUS-News-Commentary/v11/moses/en-es.txt.zip"

print("Downloading:", download_url)
subprocess.run(["wget", "-O", str(zip_path), download_url], check=True)

print("\nListing ZIP contents:")
subprocess.run(["unzip", "-l", str(zip_path)], check=True)

print("\nExtracting ZIP into:", mydrive)
subprocess.run(["unzip", "-o", str(zip_path), "-d", str(mydrive)], check=True)

print("\nFiles now present in /content/drive/MyDrive/anlp:")
for p in sorted(mydrive.iterdir()):
    print(" -", p.name)

Mounted at /content/drive
Downloading: https://object.pouta.csc.fi/OPUS-News-Commentary/v11/moses/en-es.txt.zip

Listing ZIP contents:

Extracting ZIP into: /content/drive/MyDrive/anlp

Files now present in /content/drive/MyDrive/anlp:
 - LICENSE
 - News-Commentary.en-es.en
 - News-Commentary.en-es.es
 - News-Commentary.en-es.ids
 - README
 - en-es.txt.zip
 - iterative_backtranslation_en_to_es
 - iterative_backtranslation_es_to_en
 - multiun_clean.en
 - multiun_clean.es
 - news_commentary_clean.en
 - news_commentary_clean.es
 - news_commentary_dev.en
 - news_commentary_dev.es
 - news_commentary_mono.en
 - news_commentary_mono.es
 - news_commentary_test.en
 - news_commentary_test.es
 - news_commentary_train.en
 - news_commentary_train.es


In [8]:
# =========================
# CELL 1: DRIVE + PATHS
# =========================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

# Root folder required by the assignment
mydrive = Path("/content/drive/MyDrive/anlp")
mydrive.mkdir(parents=True, exist_ok=True)

# Language pair
source = "en"
target = "es"

# ------------------------------------------------------------
# CHANGE ONLY THESE TWO VARIABLES IF THE EXTRACTED FILENAMES
# IN YOUR DRIVE ARE DIFFERENT
# ------------------------------------------------------------
raw_source_filename = "News-Commentary.en-es.en"
raw_target_filename = "News-Commentary.en-es.es"

# Cleaned output names to be used from this point onward
clean_prefix = "news_commentary_clean"
clean_source_filename = f"{clean_prefix}.{source}"
clean_target_filename = f"{clean_prefix}.{target}"

raw_source_path = mydrive / raw_source_filename
raw_target_path = mydrive / raw_target_filename

clean_source_path = mydrive / clean_source_filename
clean_target_path = mydrive / clean_target_filename

print("Raw source file :", raw_source_path)
print("Raw target file :", raw_target_path)
print("Clean source out:", clean_source_path)
print("Clean target out:", clean_target_path)

print("\nCurrent files in the anlp folder:")
for p in sorted(mydrive.iterdir()):
    print(" -", p.name)

assert raw_source_path.exists(), f"Raw source file not found: {raw_source_path}"
assert raw_target_path.exists(), f"Raw target file not found: {raw_target_path}"

Mounted at /content/drive
Raw source file : /content/drive/MyDrive/anlp/News-Commentary.en-es.en
Raw target file : /content/drive/MyDrive/anlp/News-Commentary.en-es.es
Clean source out: /content/drive/MyDrive/anlp/news_commentary_clean.en
Clean target out: /content/drive/MyDrive/anlp/news_commentary_clean.es

Current files in the anlp folder:
 - LICENSE
 - News-Commentary.en-es.en
 - News-Commentary.en-es.es
 - News-Commentary.en-es.ids
 - README
 - en-es.txt.zip
 - iterative_backtranslation_en_to_es
 - iterative_backtranslation_es_to_en
 - multiun_clean.en
 - multiun_clean.es
 - news_commentary_clean.en
 - news_commentary_clean.es
 - news_commentary_dev.en
 - news_commentary_dev.es
 - news_commentary_mono.en
 - news_commentary_mono.es
 - news_commentary_test.en
 - news_commentary_test.es
 - news_commentary_train.en
 - news_commentary_train.es


In [9]:
# =========================
# CELL 2: WRITE PROVIDED SCRIPT
# =========================

from pathlib import Path

script_path = Path("/content/preprocess-corpus.sh")

script_content = r"""#!/bin/bash

if [ $# -ne 4 ]; then
  echo "Error: Wrong number of arguments"
  echo "Usage: $0 <filein.sl> <filein.tl> <fileout.sl> <fileout.tl>"
  exit 1
fi

file_sl="$1"
file_tl="$2"

file_sl_out="$3"
file_tl_out="$4"

if [ ! -f "$file_sl" ]; then
  echo "Error: File '$file_sl' does not exist."
  exit 1
fi

if [ ! -f "$file_tl" ]; then
  echo "Error: File '$file_tl' does not exist."
  exit 1
fi

if [ -f "$file_sl_out" ]; then
  echo "Error: File '$file_sl_out' already exist."
  echo "Please remove it"
  exit 1
fi

if [ -f "$file_tl_out" ]; then
  echo "Error: File '$file_tl_out' already exist."
    echo "Please remove it"
  exit 1
fi

lines_sl=$(wc -l < "$file_sl")
lines_tl=$(wc -l < "$file_tl")

if [ "$lines_sl" -ne "$lines_tl" ]; then
  echo "Error: The files provide do not contain the same number of lines:"
  echo "   $file_sl: $lines_sl lines"
  echo "   $file_tl: $lines_tl lines"
fi

temp=$(mktemp -p "$PWD")

cat $file_sl | sed -re "s/^\s+//g" | sed -re "s/\s+$//g" | sed -re "s/\s+/ /g" > $temp"-sl"
cat $file_tl | sed -re "s/^\s+//g" | sed -re "s/\s+$//g" | sed -re "s/\s+/ /g" > $temp"-tl"

paste $temp"-sl" $temp"-tl" | sort | uniq |\
awk -F$'\t' '{if (($1!="")&&($2!="")) print}' | shuf > $temp-"sltl"

cut -f1 $temp-"sltl" > "$file_sl_out"
cut -f2 $temp-"sltl" > "$file_tl_out"

rm $temp-"sltl" $temp"-sl" $temp"-tl"
"""

script_path.write_text(script_content, encoding="utf-8")
script_path.chmod(0o755)

print(f"Script written to: {script_path}")

Script written to: /content/preprocess-corpus.sh


In [10]:
# =========================
# CELL 3: HELPERS FOR SAFE EXECUTION AND VALIDATION (CORRECTED)
# =========================

import re
import subprocess
from pathlib import Path

def count_lines(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f)

def read_lines(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

def normalize_script_like_whitespace(text):
    """
    Approximate the behavior of the provided shell script:
    - trim leading/trailing regular whitespace
    - collapse repeated whitespace to a single space

    Important:
    This is only a validator approximation and should not be stricter
    than the guarantees required by the assignment.
    """
    return re.sub(r"\s+", " ", text.strip())

def validate_clean_parallel_corpus(source_path, target_path, verbose=True):
    """
    Validate the cleaned Moses-format parallel corpus.

    Required checks for the assignment:
    1) Same number of lines on both sides
    2) No blank lines
    3) No duplicate parallel pairs
    4) Structural alignment preserved

    Optional diagnostic:
    5) Detect suspicious spacing patterns, but do not fail only because
       Unicode whitespace may remain after the shell preprocessing.
    """
    source_path = Path(source_path)
    target_path = Path(target_path)

    if not source_path.exists():
        raise FileNotFoundError(f"Clean source file not found: {source_path}")
    if not target_path.exists():
        raise FileNotFoundError(f"Clean target file not found: {target_path}")

    source_lines = read_lines(source_path)
    target_lines = read_lines(target_path)

    if len(source_lines) != len(target_lines):
        raise ValueError(
            f"Line count mismatch after preprocessing: "
            f"{source_path} has {len(source_lines)} lines, "
            f"{target_path} has {len(target_lines)} lines."
        )

    if len(source_lines) == 0:
        raise ValueError("The cleaned corpus is empty.")

    blank_source_idx = [i for i, line in enumerate(source_lines) if line.strip() == ""]
    blank_target_idx = [i for i, line in enumerate(target_lines) if line.strip() == ""]

    if blank_source_idx:
        raise ValueError(f"Blank lines found in source file at indices: {blank_source_idx[:10]}")
    if blank_target_idx:
        raise ValueError(f"Blank lines found in target file at indices: {blank_target_idx[:10]}")

    pairs = list(zip(source_lines, target_lines))
    unique_pairs = set(pairs)

    if len(unique_pairs) != len(pairs):
        raise ValueError(
            f"Duplicate parallel pairs detected after preprocessing: "
            f"{len(pairs) - len(unique_pairs)} duplicates still remain."
        )

    # Diagnostic only: detect lines that still change under generic normalization
    suspicious_source_ws = [i for i, line in enumerate(source_lines) if line != normalize_script_like_whitespace(line)]
    suspicious_target_ws = [i for i, line in enumerate(target_lines) if line != normalize_script_like_whitespace(line)]

    summary = {
        "num_pairs": len(pairs),
        "source_path": str(source_path),
        "target_path": str(target_path),
        "same_number_of_lines": True,
        "no_blank_lines": True,
        "no_duplicate_parallel_pairs": True,
        "alignment_is_structurally_consistent": True,
        "whitespace_check_note": (
            "Structural validation passed. Some lines may still contain Unicode spacing "
            "characters not fully normalized by the provided shell script."
            if suspicious_source_ws or suspicious_target_ws
            else "No suspicious spacing patterns detected."
        ),
        "num_suspicious_source_spacing_lines": len(suspicious_source_ws),
        "num_suspicious_target_spacing_lines": len(suspicious_target_ws),
    }

    if verbose:
        print("Validation successful.")
        print(f"Number of cleaned parallel pairs: {summary['num_pairs']}")
        print(f"Clean source file: {summary['source_path']}")
        print(f"Clean target file: {summary['target_path']}")
        print(summary["whitespace_check_note"])
        if suspicious_source_ws:
            print("Example suspicious source indices:", suspicious_source_ws[:10])
        if suspicious_target_ws:
            print("Example suspicious target indices:", suspicious_target_ws[:10])

    return summary

def run_preprocess_script_locally(
    script_path,
    raw_source_path,
    raw_target_path,
    clean_source_path,
    clean_target_path,
    overwrite=False,
    verbose=True,
):
    """
    Run the provided shell script on local Colab storage for speed,
    then copy the cleaned outputs back to Google Drive.

    This wrapper keeps the original shell script unchanged, but adds:
    - strict input validation before execution
    - output validation after execution
    """
    import shutil
    import time

    script_path = Path(script_path)
    raw_source_path = Path(raw_source_path)
    raw_target_path = Path(raw_target_path)
    clean_source_path = Path(clean_source_path)
    clean_target_path = Path(clean_target_path)

    if not script_path.exists():
        raise FileNotFoundError(f"Script not found: {script_path}")
    if not raw_source_path.exists():
        raise FileNotFoundError(f"Raw source file not found: {raw_source_path}")
    if not raw_target_path.exists():
        raise FileNotFoundError(f"Raw target file not found: {raw_target_path}")

    raw_source_lines = count_lines(raw_source_path)
    raw_target_lines = count_lines(raw_target_path)

    if raw_source_lines != raw_target_lines:
        raise ValueError(
            f"Raw corpus files do not have the same number of lines: "
            f"{raw_source_path} has {raw_source_lines}, "
            f"{raw_target_path} has {raw_target_lines}."
        )

    workdir = Path("/content/anlp_work")
    workdir.mkdir(parents=True, exist_ok=True)

    local_raw_source = workdir / raw_source_path.name
    local_raw_target = workdir / raw_target_path.name
    local_clean_source = workdir / clean_source_path.name
    local_clean_target = workdir / clean_target_path.name

    for p in [local_raw_source, local_raw_target, local_clean_source, local_clean_target]:
        if p.exists():
            p.unlink()

    if overwrite:
        for p in [clean_source_path, clean_target_path]:
            if p.exists():
                p.unlink()
    else:
        if clean_source_path.exists():
            raise FileExistsError(f"Output file already exists: {clean_source_path}")
        if clean_target_path.exists():
            raise FileExistsError(f"Output file already exists: {clean_target_path}")

    if verbose:
        print("Copying raw files from Google Drive to local Colab storage...")

    shutil.copy2(raw_source_path, local_raw_source)
    shutil.copy2(raw_target_path, local_raw_target)

    if verbose:
        print("Running preprocess-corpus.sh locally...")

    start = time.time()
    result = subprocess.run(
        [
            "bash",
            str(script_path),
            str(local_raw_source),
            str(local_raw_target),
            str(local_clean_source),
            str(local_clean_target),
        ],
        text=True,
        capture_output=True,
    )
    elapsed = time.time() - start

    if verbose:
        print(f"Finished in {elapsed:.2f} seconds.")
        print("Return code:", result.returncode)
        if result.stdout.strip():
            print("\n--- STDOUT ---")
            print(result.stdout)
        if result.stderr.strip():
            print("\n--- STDERR ---")
            print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            "preprocess-corpus.sh failed.\n"
            f"Return code: {result.returncode}\n"
            f"STDOUT:\n{result.stdout}\n"
            f"STDERR:\n{result.stderr}"
        )

    if verbose:
        print("Copying cleaned files back to Google Drive...")

    shutil.copy2(local_clean_source, clean_source_path)
    shutil.copy2(local_clean_target, clean_target_path)

    validation_summary = validate_clean_parallel_corpus(
        clean_source_path,
        clean_target_path,
        verbose=verbose,
    )

    return {
        "raw_source_lines": raw_source_lines,
        "raw_target_lines": raw_target_lines,
        "clean_pairs": validation_summary["num_pairs"],
        "clean_source_path": str(clean_source_path),
        "clean_target_path": str(clean_target_path),
        "num_suspicious_source_spacing_lines": validation_summary["num_suspicious_source_spacing_lines"],
        "num_suspicious_target_spacing_lines": validation_summary["num_suspicious_target_spacing_lines"],
    }

In [11]:
# =========================
# CELL 4: RUN PREPROCESSING ON THE REAL CORPUS
# =========================

summary = run_preprocess_script_locally(
    script_path=script_path,
    raw_source_path=raw_source_path,
    raw_target_path=raw_target_path,
    clean_source_path=clean_source_path,
    clean_target_path=clean_target_path,
    overwrite=True,
    verbose=True,
)

print("\nPreprocessing summary:")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\nImportant note:")
print("The provided script shuffles the cleaned parallel pairs with 'shuf',")
print("so the cleaned corpus will normally NOT preserve the original order.")

Copying raw files from Google Drive to local Colab storage...
Running preprocess-corpus.sh locally...
Finished in 15.37 seconds.
Return code: 0
Copying cleaned files back to Google Drive...
Validation successful.
Number of cleaned parallel pairs: 238511
Clean source file: /content/drive/MyDrive/anlp/news_commentary_clean.en
Clean target file: /content/drive/MyDrive/anlp/news_commentary_clean.es
Structural validation passed. Some lines may still contain Unicode spacing characters not fully normalized by the provided shell script.
Example suspicious source indices: [1213, 1350, 1514, 2303, 3262, 3469, 3609, 3699, 3916, 4444]
Example suspicious target indices: [62, 445, 465, 1675, 2205, 2279, 2471, 3441, 4017, 4021]

Preprocessing summary:
raw_source_lines: 238872
raw_target_lines: 238872
clean_pairs: 238511
clean_source_path: /content/drive/MyDrive/anlp/news_commentary_clean.en
clean_target_path: /content/drive/MyDrive/anlp/news_commentary_clean.es
num_suspicious_source_spacing_lines: 58

In [12]:
# =========================
# CELL 5: INSPECT A FEW CLEANED PAIRS
# =========================

clean_source_lines = read_lines(clean_source_path)
clean_target_lines = read_lines(clean_target_path)

print(f"Cleaned source lines: {len(clean_source_lines)}")
print(f"Cleaned target lines: {len(clean_target_lines)}")

print("\nFirst 5 cleaned parallel pairs:")
for i, (src, tgt) in enumerate(zip(clean_source_lines[:5], clean_target_lines[:5]), start=1):
    print(f"\nPair {i}")
    print("EN:", src)
    print("ES:", tgt)

Cleaned source lines: 238511
Cleaned target lines: 238511

First 5 cleaned parallel pairs:

Pair 1
EN: Arguably, Russia has improved its relations with China under Putin, but at the cost of acceding to China’s demands for two big disputed islands over which the two countries fought in 1969.
ES: Supuestamente bajo Putin Rusia ha mejorado sus relaciones con China, pero al coste de ceder a las demandas chinas de dos grandes islas en disputa, por las cuales ambos países entraron en conflicto en 1969.

Pair 2
EN: If the exit cannot be navigated successfully, a dovish Fed is more likely to blow bubbles.
ES: Si no se logra una salida exitosa, lo más probable es que una postura conciliadora y abierta a las negociaciones de la Fed implique burbujas.

Pair 3
EN: For the West – and for Europe first and foremost – this dilemma cannot be avoided.
ES: Para Occidente –y ante todo para Europa-, este dilema no se puede evitar.

Pair 4
EN: One problem is that workers have fixed costs, such as transporta

In [13]:
# =========================
# CELL 6: EXPLICIT QUALITY CHECKS AFTER PREPROCESSING
# =========================

clean_source_lines = read_lines(clean_source_path)
clean_target_lines = read_lines(clean_target_path)
clean_pairs = list(zip(clean_source_lines, clean_target_lines))

same_num_lines = len(clean_source_lines) == len(clean_target_lines)
no_blank_lines = all(line.strip() != "" for line in clean_source_lines) and \
                 all(line.strip() != "" for line in clean_target_lines)
no_duplicate_pairs = len(clean_pairs) == len(set(clean_pairs))

print("QUALITY CHECKS AFTER PREPROCESSING")
print("----------------------------------")
print(f"1. Same number of lines on both sides: {same_num_lines}")
print(f"2. No blank lines: {no_blank_lines}")
print(f"3. No duplicate parallel pairs: {no_duplicate_pairs}")
print(f"4. Structural alignment preserved: {same_num_lines and len(clean_pairs) == len(clean_source_lines)}")

assert same_num_lines, "The cleaned source and target files do not contain the same number of lines."
assert no_blank_lines, "Blank lines were found after preprocessing."
assert no_duplicate_pairs, "Duplicate parallel pairs remain after preprocessing."

print("\nAll required quality checks passed.")
print("Note: minor Unicode spacing characters may still exist in some lines,")
print("but the core preprocessing guarantees required by the assignment are satisfied.")

QUALITY CHECKS AFTER PREPROCESSING
----------------------------------
1. Same number of lines on both sides: True
2. No blank lines: True
3. No duplicate parallel pairs: True
4. Structural alignment preserved: True

All required quality checks passed.
Note: minor Unicode spacing characters may still exist in some lines,
but the core preprocessing guarantees required by the assignment are satisfied.


In [14]:
# =========================
# CELL 7: UNIT TESTS
# =========================

import tempfile
import unittest

class TestCorpusPreprocessing(unittest.TestCase):

    def _write_lines(self, path, lines):
        Path(path).write_text("".join(lines), encoding="utf-8")

    def test_successful_cleaning_removes_duplicates_empties_and_normalizes_spacing(self):
        with tempfile.TemporaryDirectory() as tmpdir:
            tmpdir = Path(tmpdir)

            raw_en = tmpdir / "raw.en"
            raw_es = tmpdir / "raw.es"
            clean_en = tmpdir / "clean.en"
            clean_es = tmpdir / "clean.es"

            en_lines = [
                "  Hello   world  \n",
                "Hello world\n",
                "Keep   me  \n",
                "   \n",
                "Pair with empty target\n",
                "Unique sentence\n",
                "Hello world!\n",
            ]

            es_lines = [
                " Hola   mundo \n",
                "Hola mundo\n",
                " Mantenme \n",
                "Esto se elimina\n",
                "   \n",
                "Oración única\n",
                "¡Hola mundo!\n",
            ]

            self._write_lines(raw_en, en_lines)
            self._write_lines(raw_es, es_lines)

            summary = run_preprocess_script_locally(
                script_path=script_path,
                raw_source_path=raw_en,
                raw_target_path=raw_es,
                clean_source_path=clean_en,
                clean_target_path=clean_es,
                overwrite=False,
                verbose=False,
            )

            self.assertEqual(summary["raw_source_lines"], 7)
            self.assertEqual(summary["raw_target_lines"], 7)

            out_en = read_lines(clean_en)
            out_es = read_lines(clean_es)
            out_pairs = set(zip(out_en, out_es))

            expected_pairs = {
                ("Hello world", "Hola mundo"),
                ("Keep me", "Mantenme"),
                ("Unique sentence", "Oración única"),
                ("Hello world!", "¡Hola mundo!"),
            }

            self.assertEqual(out_pairs, expected_pairs)
            self.assertEqual(len(out_en), 4)
            self.assertEqual(len(out_es), 4)
            self.assertTrue(all(line == normalize_script_like_whitespace(line) for line in out_en))
            self.assertTrue(all(line == normalize_script_like_whitespace(line) for line in out_es))

    def test_wrapper_rejects_raw_line_count_mismatch(self):
        with tempfile.TemporaryDirectory() as tmpdir:
            tmpdir = Path(tmpdir)

            raw_en = tmpdir / "raw.en"
            raw_es = tmpdir / "raw.es"
            clean_en = tmpdir / "clean.en"
            clean_es = tmpdir / "clean.es"

            self._write_lines(raw_en, ["A\n", "B\n"])
            self._write_lines(raw_es, ["Uno\n"])

            with self.assertRaises(ValueError):
                run_preprocess_script_locally(
                    script_path=script_path,
                    raw_source_path=raw_en,
                    raw_target_path=raw_es,
                    clean_source_path=clean_en,
                    clean_target_path=clean_es,
                    overwrite=False,
                    verbose=False,
                )

    def test_validator_detects_duplicate_pairs(self):
        with tempfile.TemporaryDirectory() as tmpdir:
            tmpdir = Path(tmpdir)

            clean_en = tmpdir / "clean.en"
            clean_es = tmpdir / "clean.es"

            self._write_lines(clean_en, ["Hello world\n", "Hello world\n"])
            self._write_lines(clean_es, ["Hola mundo\n", "Hola mundo\n"])

            with self.assertRaises(ValueError):
                validate_clean_parallel_corpus(clean_en, clean_es, verbose=False)

    def test_validator_detects_blank_lines(self):
        with tempfile.TemporaryDirectory() as tmpdir:
            tmpdir = Path(tmpdir)

            clean_en = tmpdir / "clean.en"
            clean_es = tmpdir / "clean.es"

            self._write_lines(clean_en, ["Hello world\n", "\n"])
            self._write_lines(clean_es, ["Hola mundo\n", "Segunda línea\n"])

            with self.assertRaises(ValueError):
                validate_clean_parallel_corpus(clean_en, clean_es, verbose=False)

    def test_overwrite_true_allows_rerun(self):
        with tempfile.TemporaryDirectory() as tmpdir:
            tmpdir = Path(tmpdir)

            raw_en = tmpdir / "raw.en"
            raw_es = tmpdir / "raw.es"
            clean_en = tmpdir / "clean.en"
            clean_es = tmpdir / "clean.es"

            self._write_lines(raw_en, ["A\n", "B\n"])
            self._write_lines(raw_es, ["Uno\n", "Dos\n"])

            run_preprocess_script_locally(
                script_path=script_path,
                raw_source_path=raw_en,
                raw_target_path=raw_es,
                clean_source_path=clean_en,
                clean_target_path=clean_es,
                overwrite=False,
                verbose=False,
            )

            run_preprocess_script_locally(
                script_path=script_path,
                raw_source_path=raw_en,
                raw_target_path=raw_es,
                clean_source_path=clean_en,
                clean_target_path=clean_es,
                overwrite=True,
                verbose=False,
            )

            self.assertTrue(clean_en.exists())
            self.assertTrue(clean_es.exists())

unittest.main(argv=[""], verbosity=2, exit=False)

test_overwrite_true_allows_rerun (__main__.TestCorpusPreprocessing.test_overwrite_true_allows_rerun) ... ok
test_successful_cleaning_removes_duplicates_empties_and_normalizes_spacing (__main__.TestCorpusPreprocessing.test_successful_cleaning_removes_duplicates_empties_and_normalizes_spacing) ... ok
test_validator_detects_blank_lines (__main__.TestCorpusPreprocessing.test_validator_detects_blank_lines) ... ok
test_validator_detects_duplicate_pairs (__main__.TestCorpusPreprocessing.test_validator_detects_duplicate_pairs) ... ok
test_wrapper_rejects_raw_line_count_mismatch (__main__.TestCorpusPreprocessing.test_wrapper_rejects_raw_line_count_mismatch) ... ok
test_bleu_metric_is_available (__main__.TestEnvironmentSetupAndReproducibility.test_bleu_metric_is_available) ... FAIL
test_expected_assignment_sizes_are_respected (__main__.TestEnvironmentSetupAndReproducibility.test_expected_assignment_sizes_are_respected) ... ok
test_output_directories_exist (__main__.TestEnvironmentSetupAndReprodu


no tests ran in 0.04s


## SECTION 7

In [15]:
# =========================
# CELL 8: DEFINE SPLIT FILES AND REQUIRED SIZES
# =========================

# Exact sizes required by the assignment
parallel_training_size = 1000
development_parallel_size = 200
test_parallel_size = 200
english_monolingual_size = 1000
spanish_monolingual_size = 1000

# Output filenames for the final experimental subsets
parallel_training_source_filename = "news_commentary_train.en"
parallel_training_target_filename = "news_commentary_train.es"

development_source_filename = "news_commentary_dev.en"
development_target_filename = "news_commentary_dev.es"

test_source_filename = "news_commentary_test.en"
test_target_filename = "news_commentary_test.es"

english_monolingual_filename = "news_commentary_mono.en"
spanish_monolingual_filename = "news_commentary_mono.es"

# Output paths
parallel_training_source_path = mydrive / parallel_training_source_filename
parallel_training_target_path = mydrive / parallel_training_target_filename

development_source_path = mydrive / development_source_filename
development_target_path = mydrive / development_target_filename

test_source_path = mydrive / test_source_filename
test_target_path = mydrive / test_target_filename

english_monolingual_path = mydrive / english_monolingual_filename
spanish_monolingual_path = mydrive / spanish_monolingual_filename

print("Final split files that will be created:")
print("Parallel training EN:", parallel_training_source_path)
print("Parallel training ES:", parallel_training_target_path)
print("Development EN      :", development_source_path)
print("Development ES      :", development_target_path)
print("Test EN             :", test_source_path)
print("Test ES             :", test_target_path)
print("Monolingual EN      :", english_monolingual_path)
print("Monolingual ES      :", spanish_monolingual_path)

print("\nRequired sizes:")
print("Parallel training size :", parallel_training_size)
print("Development size       :", development_parallel_size)
print("Test size              :", test_parallel_size)
print("English monolingual    :", english_monolingual_size)
print("Spanish monolingual    :", spanish_monolingual_size)

Final split files that will be created:
Parallel training EN: /content/drive/MyDrive/anlp/news_commentary_train.en
Parallel training ES: /content/drive/MyDrive/anlp/news_commentary_train.es
Development EN      : /content/drive/MyDrive/anlp/news_commentary_dev.en
Development ES      : /content/drive/MyDrive/anlp/news_commentary_dev.es
Test EN             : /content/drive/MyDrive/anlp/news_commentary_test.en
Test ES             : /content/drive/MyDrive/anlp/news_commentary_test.es
Monolingual EN      : /content/drive/MyDrive/anlp/news_commentary_mono.en
Monolingual ES      : /content/drive/MyDrive/anlp/news_commentary_mono.es

Required sizes:
Parallel training size : 1000
Development size       : 200
Test size              : 200
English monolingual    : 1000
Spanish monolingual    : 1000


In [16]:
# =========================
# CELL 9: SPLIT HELPERS AND NON-OVERLAP VALIDATION
# =========================

from pathlib import Path

def write_lines_to_file(output_path, lines):
    output_path = Path(output_path)
    with output_path.open("w", encoding="utf-8") as f:
        for line in lines:
            f.write(f"{line}\n")

def write_parallel_pairs_to_moses_files(source_output_path, target_output_path, parallel_pairs):
    source_lines = [source_sentence for source_sentence, _ in parallel_pairs]
    target_lines = [target_sentence for _, target_sentence in parallel_pairs]
    write_lines_to_file(source_output_path, source_lines)
    write_lines_to_file(target_output_path, target_lines)

def ensure_output_paths_are_writable(output_paths, overwrite=False):
    for output_path in output_paths:
        output_path = Path(output_path)
        if output_path.exists():
            if overwrite:
                output_path.unlink()
            else:
                raise FileExistsError(
                    f"Output file already exists: {output_path}. "
                    f"Set overwrite=True if you want to recreate it."
                )

def read_parallel_pairs_from_moses_files(source_path, target_path):
    source_lines = read_lines(source_path)
    target_lines = read_lines(target_path)

    if len(source_lines) != len(target_lines):
        raise ValueError(
            f"Moses files do not have the same number of lines: "
            f"{source_path} has {len(source_lines)}, "
            f"{target_path} has {len(target_lines)}."
        )

    return list(zip(source_lines, target_lines))

def build_experimental_split_from_clean_corpus(
    clean_source_corpus_path,
    clean_target_corpus_path,
    parallel_training_source_output_path,
    parallel_training_target_output_path,
    development_source_output_path,
    development_target_output_path,
    test_source_output_path,
    test_target_output_path,
    english_monolingual_output_path,
    spanish_monolingual_output_path,
    parallel_training_size,
    development_parallel_size,
    test_parallel_size,
    english_monolingual_size,
    spanish_monolingual_size,
    overwrite=False,
    verbose=True,
):
    """
    Build the final experimental split from the cleaned bilingual corpus.

    Design policy:
    - Parallel train/dev/test are taken as disjoint sentence-pair slices
      from the cleaned bilingual pool.
    - Monolingual English and Spanish sets are extracted only from the
      remaining part of the cleaned corpus.
    - English monolingual sentences must not appear on the English side
      of any parallel subset.
    - Spanish monolingual sentences must not appear on the Spanish side
      of any parallel subset.
    """

    clean_source_corpus_path = Path(clean_source_corpus_path)
    clean_target_corpus_path = Path(clean_target_corpus_path)

    cleaned_source_sentences = read_lines(clean_source_corpus_path)
    cleaned_target_sentences = read_lines(clean_target_corpus_path)

    if len(cleaned_source_sentences) != len(cleaned_target_sentences):
        raise ValueError(
            f"The cleaned corpus files do not have the same number of lines: "
            f"{clean_source_corpus_path} has {len(cleaned_source_sentences)}, "
            f"{clean_target_corpus_path} has {len(cleaned_target_sentences)}."
        )

    cleaned_parallel_pairs = list(zip(cleaned_source_sentences, cleaned_target_sentences))

    required_parallel_pairs = (
        parallel_training_size
        + development_parallel_size
        + test_parallel_size
    )

    if len(cleaned_parallel_pairs) < required_parallel_pairs:
        raise ValueError(
            f"The cleaned corpus is too small. "
            f"At least {required_parallel_pairs} cleaned parallel pairs are required "
            f"for train/dev/test, but only {len(cleaned_parallel_pairs)} are available."
        )

    ensure_output_paths_are_writable(
        [
            parallel_training_source_output_path,
            parallel_training_target_output_path,
            development_source_output_path,
            development_target_output_path,
            test_source_output_path,
            test_target_output_path,
            english_monolingual_output_path,
            spanish_monolingual_output_path,
        ],
        overwrite=overwrite,
    )

    # Because the provided preprocessing script already shuffles the cleaned pool,
    # taking consecutive disjoint slices is a simple and reproducible split policy.
    parallel_training_end = parallel_training_size
    development_end = parallel_training_end + development_parallel_size
    test_end = development_end + test_parallel_size

    parallel_training_pairs = cleaned_parallel_pairs[:parallel_training_end]
    development_pairs = cleaned_parallel_pairs[parallel_training_end:development_end]
    test_pairs = cleaned_parallel_pairs[development_end:test_end]

    remaining_pairs_for_monolingual_extraction = cleaned_parallel_pairs[test_end:]

    # Sentence-level exclusion sets for monolingual extraction
    parallel_source_sentences = {
        source_sentence
        for source_sentence, _ in (parallel_training_pairs + development_pairs + test_pairs)
    }
    parallel_target_sentences = {
        target_sentence
        for _, target_sentence in (parallel_training_pairs + development_pairs + test_pairs)
    }

    english_monolingual_sentences = []
    seen_english_monolingual_sentences = set()

    for source_sentence, _ in remaining_pairs_for_monolingual_extraction:
        if source_sentence.strip() == "":
            continue
        if source_sentence in parallel_source_sentences:
            continue
        if source_sentence in seen_english_monolingual_sentences:
            continue

        english_monolingual_sentences.append(source_sentence)
        seen_english_monolingual_sentences.add(source_sentence)

        if len(english_monolingual_sentences) == english_monolingual_size:
            break

    spanish_monolingual_sentences = []
    seen_spanish_monolingual_sentences = set()

    for _, target_sentence in remaining_pairs_for_monolingual_extraction:
        if target_sentence.strip() == "":
            continue
        if target_sentence in parallel_target_sentences:
            continue
        if target_sentence in seen_spanish_monolingual_sentences:
            continue

        spanish_monolingual_sentences.append(target_sentence)
        seen_spanish_monolingual_sentences.add(target_sentence)

        if len(spanish_monolingual_sentences) == spanish_monolingual_size:
            break

    if len(english_monolingual_sentences) != english_monolingual_size:
        raise ValueError(
            f"Could not extract {english_monolingual_size} English monolingual sentences "
            f"without overlap. Only {len(english_monolingual_sentences)} were found."
        )

    if len(spanish_monolingual_sentences) != spanish_monolingual_size:
        raise ValueError(
            f"Could not extract {spanish_monolingual_size} Spanish monolingual sentences "
            f"without overlap. Only {len(spanish_monolingual_sentences)} were found."
        )

    write_parallel_pairs_to_moses_files(
        parallel_training_source_output_path,
        parallel_training_target_output_path,
        parallel_training_pairs,
    )
    write_parallel_pairs_to_moses_files(
        development_source_output_path,
        development_target_output_path,
        development_pairs,
    )
    write_parallel_pairs_to_moses_files(
        test_source_output_path,
        test_target_output_path,
        test_pairs,
    )

    write_lines_to_file(english_monolingual_output_path, english_monolingual_sentences)
    write_lines_to_file(spanish_monolingual_output_path, spanish_monolingual_sentences)

    split_summary = validate_experimental_split_non_overlap(
        parallel_training_source_output_path,
        parallel_training_target_output_path,
        development_source_output_path,
        development_target_output_path,
        test_source_output_path,
        test_target_output_path,
        english_monolingual_output_path,
        spanish_monolingual_output_path,
        expected_parallel_training_size=parallel_training_size,
        expected_development_size=development_parallel_size,
        expected_test_size=test_parallel_size,
        expected_english_monolingual_size=english_monolingual_size,
        expected_spanish_monolingual_size=spanish_monolingual_size,
        verbose=verbose,
    )

    return split_summary

def validate_experimental_split_non_overlap(
    parallel_training_source_path,
    parallel_training_target_path,
    development_source_path,
    development_target_path,
    test_source_path,
    test_target_path,
    english_monolingual_path,
    spanish_monolingual_path,
    expected_parallel_training_size,
    expected_development_size,
    expected_test_size,
    expected_english_monolingual_size,
    expected_spanish_monolingual_size,
    verbose=True,
):
    """
    Validate the final experimental split:
    - exact sizes
    - no overlap between parallel train/dev/test
    - no overlap between monolingual sentences and any parallel subset
    - no blank lines
    - no duplicate lines inside monolingual sets
    """

    parallel_training_pairs = read_parallel_pairs_from_moses_files(
        parallel_training_source_path, parallel_training_target_path
    )
    development_pairs = read_parallel_pairs_from_moses_files(
        development_source_path, development_target_path
    )
    test_pairs = read_parallel_pairs_from_moses_files(
        test_source_path, test_target_path
    )

    english_monolingual_sentences = read_lines(english_monolingual_path)
    spanish_monolingual_sentences = read_lines(spanish_monolingual_path)

    if len(parallel_training_pairs) != expected_parallel_training_size:
        raise ValueError(
            f"Parallel training set has {len(parallel_training_pairs)} pairs, "
            f"but {expected_parallel_training_size} were expected."
        )
    if len(development_pairs) != expected_development_size:
        raise ValueError(
            f"Development set has {len(development_pairs)} pairs, "
            f"but {expected_development_size} were expected."
        )
    if len(test_pairs) != expected_test_size:
        raise ValueError(
            f"Test set has {len(test_pairs)} pairs, "
            f"but {expected_test_size} were expected."
        )
    if len(english_monolingual_sentences) != expected_english_monolingual_size:
        raise ValueError(
            f"English monolingual set has {len(english_monolingual_sentences)} sentences, "
            f"but {expected_english_monolingual_size} were expected."
        )
    if len(spanish_monolingual_sentences) != expected_spanish_monolingual_size:
        raise ValueError(
            f"Spanish monolingual set has {len(spanish_monolingual_sentences)} sentences, "
            f"but {expected_spanish_monolingual_size} were expected."
        )

    if any(sentence.strip() == "" for sentence in english_monolingual_sentences):
        raise ValueError("The English monolingual file contains blank lines.")
    if any(sentence.strip() == "" for sentence in spanish_monolingual_sentences):
        raise ValueError("The Spanish monolingual file contains blank lines.")

    if len(english_monolingual_sentences) != len(set(english_monolingual_sentences)):
        raise ValueError("The English monolingual file contains duplicate sentences.")
    if len(spanish_monolingual_sentences) != len(set(spanish_monolingual_sentences)):
        raise ValueError("The Spanish monolingual file contains duplicate sentences.")

    parallel_training_pair_set = set(parallel_training_pairs)
    development_pair_set = set(development_pairs)
    test_pair_set = set(test_pairs)

    training_development_pair_overlap = parallel_training_pair_set & development_pair_set
    training_test_pair_overlap = parallel_training_pair_set & test_pair_set
    development_test_pair_overlap = development_pair_set & test_pair_set

    if training_development_pair_overlap:
        raise ValueError("Parallel training and development sets overlap.")
    if training_test_pair_overlap:
        raise ValueError("Parallel training and test sets overlap.")
    if development_test_pair_overlap:
        raise ValueError("Development and test sets overlap.")

    all_parallel_source_sentences = {
        source_sentence
        for source_sentence, _ in (parallel_training_pairs + development_pairs + test_pairs)
    }
    all_parallel_target_sentences = {
        target_sentence
        for _, target_sentence in (parallel_training_pairs + development_pairs + test_pairs)
    }

    english_monolingual_parallel_overlap = set(english_monolingual_sentences) & all_parallel_source_sentences
    spanish_monolingual_parallel_overlap = set(spanish_monolingual_sentences) & all_parallel_target_sentences

    if english_monolingual_parallel_overlap:
        raise ValueError(
            "The English monolingual set overlaps with the English side of the parallel subsets."
        )
    if spanish_monolingual_parallel_overlap:
        raise ValueError(
            "The Spanish monolingual set overlaps with the Spanish side of the parallel subsets."
        )

    validation_summary = {
        "parallel_training_pairs": len(parallel_training_pairs),
        "development_pairs": len(development_pairs),
        "test_pairs": len(test_pairs),
        "english_monolingual_sentences": len(english_monolingual_sentences),
        "spanish_monolingual_sentences": len(spanish_monolingual_sentences),
        "training_vs_development_overlap": False,
        "training_vs_test_overlap": False,
        "development_vs_test_overlap": False,
        "english_monolingual_vs_parallel_overlap": False,
        "spanish_monolingual_vs_parallel_overlap": False,
        "no_blank_lines_in_monolingual_files": True,
        "no_duplicates_in_monolingual_files": True,
    }

    if verbose:
        print("Experimental split validation successful.")
        print("Parallel training pairs      :", validation_summary["parallel_training_pairs"])
        print("Development pairs            :", validation_summary["development_pairs"])
        print("Test pairs                   :", validation_summary["test_pairs"])
        print("English monolingual sentences:", validation_summary["english_monolingual_sentences"])
        print("Spanish monolingual sentences:", validation_summary["spanish_monolingual_sentences"])
        print("\nStrict non-overlap policy satisfied across all required subsets.")

    return validation_summary

In [17]:
# =========================
# CELL 10: BUILD THE FINAL EXPERIMENTAL SPLIT FILES
# =========================

experimental_split_summary = build_experimental_split_from_clean_corpus(
    clean_source_corpus_path=clean_source_path,
    clean_target_corpus_path=clean_target_path,
    parallel_training_source_output_path=parallel_training_source_path,
    parallel_training_target_output_path=parallel_training_target_path,
    development_source_output_path=development_source_path,
    development_target_output_path=development_target_path,
    test_source_output_path=test_source_path,
    test_target_output_path=test_target_path,
    english_monolingual_output_path=english_monolingual_path,
    spanish_monolingual_output_path=spanish_monolingual_path,
    parallel_training_size=parallel_training_size,
    development_parallel_size=development_parallel_size,
    test_parallel_size=test_parallel_size,
    english_monolingual_size=english_monolingual_size,
    spanish_monolingual_size=spanish_monolingual_size,
    overwrite=True,
    verbose=True,
)

print("\nExperimental split summary:")
for key, value in experimental_split_summary.items():
    print(f"{key}: {value}")

Experimental split validation successful.
Parallel training pairs      : 1000
Development pairs            : 200
Test pairs                   : 200
English monolingual sentences: 1000
Spanish monolingual sentences: 1000

Strict non-overlap policy satisfied across all required subsets.

Experimental split summary:
parallel_training_pairs: 1000
development_pairs: 200
test_pairs: 200
english_monolingual_sentences: 1000
spanish_monolingual_sentences: 1000
training_vs_development_overlap: False
training_vs_test_overlap: False
development_vs_test_overlap: False
english_monolingual_vs_parallel_overlap: False
spanish_monolingual_vs_parallel_overlap: False
no_blank_lines_in_monolingual_files: True
no_duplicates_in_monolingual_files: True


In [18]:
# =========================
# CELL 11: INSPECT FINAL SPLITS
# =========================

print("Created files:")
print(" -", parallel_training_source_path.name)
print(" -", parallel_training_target_path.name)
print(" -", development_source_path.name)
print(" -", development_target_path.name)
print(" -", test_source_path.name)
print(" -", test_target_path.name)
print(" -", english_monolingual_path.name)
print(" -", spanish_monolingual_path.name)

parallel_training_pairs_preview = read_parallel_pairs_from_moses_files(
    parallel_training_source_path, parallel_training_target_path
)
development_pairs_preview = read_parallel_pairs_from_moses_files(
    development_source_path, development_target_path
)
test_pairs_preview = read_parallel_pairs_from_moses_files(
    test_source_path, test_target_path
)

english_monolingual_preview = read_lines(english_monolingual_path)
spanish_monolingual_preview = read_lines(spanish_monolingual_path)

print("\nFirst 2 parallel training pairs:")
for pair_index, (source_sentence, target_sentence) in enumerate(parallel_training_pairs_preview[:2], start=1):
    print(f"\nTraining pair {pair_index}")
    print("EN:", source_sentence)
    print("ES:", target_sentence)

print("\nFirst 2 development pairs:")
for pair_index, (source_sentence, target_sentence) in enumerate(development_pairs_preview[:2], start=1):
    print(f"\nDevelopment pair {pair_index}")
    print("EN:", source_sentence)
    print("ES:", target_sentence)

print("\nFirst 2 test pairs:")
for pair_index, (source_sentence, target_sentence) in enumerate(test_pairs_preview[:2], start=1):
    print(f"\nTest pair {pair_index}")
    print("EN:", source_sentence)
    print("ES:", target_sentence)

print("\nFirst 3 English monolingual sentences:")
for sentence_index, source_sentence in enumerate(english_monolingual_preview[:3], start=1):
    print(f"{sentence_index}. {source_sentence}")

print("\nFirst 3 Spanish monolingual sentences:")
for sentence_index, target_sentence in enumerate(spanish_monolingual_preview[:3], start=1):
    print(f"{sentence_index}. {target_sentence}")

print("\nPolicy reminder:")
print("- Parallel training, development, and test are pairwise disjoint.")
print("- English monolingual sentences do not appear in the English side of any parallel subset.")
print("- Spanish monolingual sentences do not appear in the Spanish side of any parallel subset.")

Created files:
 - news_commentary_train.en
 - news_commentary_train.es
 - news_commentary_dev.en
 - news_commentary_dev.es
 - news_commentary_test.en
 - news_commentary_test.es
 - news_commentary_mono.en
 - news_commentary_mono.es

First 2 parallel training pairs:

Training pair 1
EN: Arguably, Russia has improved its relations with China under Putin, but at the cost of acceding to China’s demands for two big disputed islands over which the two countries fought in 1969.
ES: Supuestamente bajo Putin Rusia ha mejorado sus relaciones con China, pero al coste de ceder a las demandas chinas de dos grandes islas en disputa, por las cuales ambos países entraron en conflicto en 1969.

Training pair 2
EN: If the exit cannot be navigated successfully, a dovish Fed is more likely to blow bubbles.
ES: Si no se logra una salida exitosa, lo más probable es que una postura conciliadora y abierta a las negociaciones de la Fed implique burbujas.

First 2 development pairs:

Development pair 1
EN: Since

In [27]:
# =========================
# CELL 12: UNIT TESTS FOR SPLIT DESIGN AND NON-OVERLAP
# =========================

import tempfile
import unittest

class TestExperimentalSplitDesign(unittest.TestCase):

    def _write_lines(self, path, lines):
        Path(path).write_text("".join(lines), encoding="utf-8")

    def test_successful_split_has_exact_sizes_and_strict_non_overlap(self):
        with tempfile.TemporaryDirectory() as temp_directory:
            temp_directory = Path(temp_directory)

            clean_source = temp_directory / "clean.en"
            clean_target = temp_directory / "clean.es"

            clean_source_lines = [
                "train_en_1\n",
                "train_en_2\n",
                "dev_en_1\n",
                "test_en_1\n",
                "train_en_1\n",          # duplicate source sentence with training -> must not enter EN mono
                "mono_en_1\n",
                "mono_en_2\n",
                "mono_en_3\n",
            ]

            clean_target_lines = [
                "train_es_1\n",
                "train_es_2\n",
                "dev_es_1\n",
                "test_es_1\n",
                "mono_es_candidate_1\n",
                "train_es_2\n",          # duplicate target sentence with training -> must not enter ES mono
                "mono_es_1\n",
                "mono_es_2\n",
            ]

            self._write_lines(clean_source, clean_source_lines)
            self._write_lines(clean_target, clean_target_lines)

            train_en = temp_directory / "train.en"
            train_es = temp_directory / "train.es"
            dev_en = temp_directory / "dev.en"
            dev_es = temp_directory / "dev.es"
            test_en = temp_directory / "test.en"
            test_es = temp_directory / "test.es"
            mono_en = temp_directory / "mono.en"
            mono_es = temp_directory / "mono.es"

            summary = build_experimental_split_from_clean_corpus(
                clean_source_corpus_path=clean_source,
                clean_target_corpus_path=clean_target,
                parallel_training_source_output_path=train_en,
                parallel_training_target_output_path=train_es,
                development_source_output_path=dev_en,
                development_target_output_path=dev_es,
                test_source_output_path=test_en,
                test_target_output_path=test_es,
                english_monolingual_output_path=mono_en,
                spanish_monolingual_output_path=mono_es,
                parallel_training_size=2,
                development_parallel_size=1,
                test_parallel_size=1,
                english_monolingual_size=2,
                spanish_monolingual_size=2,
                overwrite=False,
                verbose=False,
            )

            self.assertEqual(summary["parallel_training_pairs"], 2)
            self.assertEqual(summary["development_pairs"], 1)
            self.assertEqual(summary["test_pairs"], 1)
            self.assertEqual(summary["english_monolingual_sentences"], 2)
            self.assertEqual(summary["spanish_monolingual_sentences"], 2)

            train_pairs = set(read_parallel_pairs_from_moses_files(train_en, train_es))
            dev_pairs = set(read_parallel_pairs_from_moses_files(dev_en, dev_es))
            test_pairs = set(read_parallel_pairs_from_moses_files(test_en, test_es))
            english_monolingual_sentences = set(read_lines(mono_en))
            spanish_monolingual_sentences = set(read_lines(mono_es))

            self.assertTrue(train_pairs.isdisjoint(dev_pairs))
            self.assertTrue(train_pairs.isdisjoint(test_pairs))
            self.assertTrue(dev_pairs.isdisjoint(test_pairs))

            all_parallel_english = {source_sentence for source_sentence, _ in train_pairs | dev_pairs | test_pairs}
            all_parallel_spanish = {target_sentence for _, target_sentence in train_pairs | dev_pairs | test_pairs}

            self.assertTrue(english_monolingual_sentences.isdisjoint(all_parallel_english))
            self.assertTrue(spanish_monolingual_sentences.isdisjoint(all_parallel_spanish))

    def test_split_fails_if_clean_corpus_is_too_small_for_parallel_subsets(self):
        with tempfile.TemporaryDirectory() as temp_directory:
            temp_directory = Path(temp_directory)

            clean_source = temp_directory / "clean.en"
            clean_target = temp_directory / "clean.es"

            self._write_lines(clean_source, ["a\n", "b\n", "c\n"])
            self._write_lines(clean_target, ["uno\n", "dos\n", "tres\n"])

            with self.assertRaises(ValueError):
                build_experimental_split_from_clean_corpus(
                    clean_source_corpus_path=clean_source,
                    clean_target_corpus_path=clean_target,
                    parallel_training_source_output_path=temp_directory / "train.en",
                    parallel_training_target_output_path=temp_directory / "train.es",
                    development_source_output_path=temp_directory / "dev.en",
                    development_target_output_path=temp_directory / "dev.es",
                    test_source_output_path=temp_directory / "test.en",
                    test_target_output_path=temp_directory / "test.es",
                    english_monolingual_output_path=temp_directory / "mono.en",
                    spanish_monolingual_output_path=temp_directory / "mono.es",
                    parallel_training_size=2,
                    development_parallel_size=1,
                    test_parallel_size=1,
                    english_monolingual_size=1,
                    spanish_monolingual_size=1,
                    overwrite=False,
                    verbose=False,
                )

    def test_split_fails_if_not_enough_unique_monolingual_sentences_remain(self):
        with tempfile.TemporaryDirectory() as temp_directory:
            temp_directory = Path(temp_directory)

            clean_source = temp_directory / "clean.en"
            clean_target = temp_directory / "clean.es"

            clean_source_lines = [
                "train_en_1\n",
                "dev_en_1\n",
                "test_en_1\n",
                "train_en_1\n",   # excluded from EN mono because it appears in parallel training
                "dev_en_1\n",     # excluded from EN mono because it appears in dev
            ]

            clean_target_lines = [
                "train_es_1\n",
                "dev_es_1\n",
                "test_es_1\n",
                "train_es_1\n",
                "dev_es_1\n",
            ]

            self._write_lines(clean_source, clean_source_lines)
            self._write_lines(clean_target, clean_target_lines)

            with self.assertRaises(ValueError):
                build_experimental_split_from_clean_corpus(
                    clean_source_corpus_path=clean_source,
                    clean_target_corpus_path=clean_target,
                    parallel_training_source_output_path=temp_directory / "train.en",
                    parallel_training_target_output_path=temp_directory / "train.es",
                    development_source_output_path=temp_directory / "dev.en",
                    development_target_output_path=temp_directory / "dev.es",
                    test_source_output_path=temp_directory / "test.en",
                    test_target_output_path=temp_directory / "test.es",
                    english_monolingual_output_path=temp_directory / "mono.en",
                    spanish_monolingual_output_path=temp_directory / "mono.es",
                    parallel_training_size=1,
                    development_parallel_size=1,
                    test_parallel_size=1,
                    english_monolingual_size=1,
                    spanish_monolingual_size=1,
                    overwrite=False,
                    verbose=False,
                )

    def test_validation_detects_monolingual_overlap_with_parallel_sentences(self):
        with tempfile.TemporaryDirectory() as temp_directory:
            temp_directory = Path(temp_directory)

            train_en = temp_directory / "train.en"
            train_es = temp_directory / "train.es"
            dev_en = temp_directory / "dev.en"
            dev_es = temp_directory / "dev.es"
            test_en = temp_directory / "test.en"
            test_es = temp_directory / "test.es"
            mono_en = temp_directory / "mono.en"
            mono_es = temp_directory / "mono.es"

            self._write_lines(train_en, ["shared_en\n"])
            self._write_lines(train_es, ["shared_es\n"])
            self._write_lines(dev_en, ["dev_en\n"])
            self._write_lines(dev_es, ["dev_es\n"])
            self._write_lines(test_en, ["test_en\n"])
            self._write_lines(test_es, ["test_es\n"])
            self._write_lines(mono_en, ["shared_en\n"])   # invalid overlap
            self._write_lines(mono_es, ["mono_es_only\n"])

            with self.assertRaises(ValueError):
                validate_experimental_split_non_overlap(
                    parallel_training_source_path=train_en,
                    parallel_training_target_path=train_es,
                    development_source_path=dev_en,
                    development_target_path=dev_es,
                    test_source_path=test_en,
                    test_target_path=test_es,
                    english_monolingual_path=mono_en,
                    spanish_monolingual_path=mono_es,
                    expected_parallel_training_size=1,
                    expected_development_size=1,
                    expected_test_size=1,
                    expected_english_monolingual_size=1,
                    expected_spanish_monolingual_size=1,
                    verbose=False,
                )

unittest.main(argv=[""], verbosity=2, exit=False)

test_overwrite_true_allows_rerun (__main__.TestCorpusPreprocessing.test_overwrite_true_allows_rerun) ... ok
test_successful_cleaning_removes_duplicates_empties_and_normalizes_spacing (__main__.TestCorpusPreprocessing.test_successful_cleaning_removes_duplicates_empties_and_normalizes_spacing) ... ok
test_validator_detects_blank_lines (__main__.TestCorpusPreprocessing.test_validator_detects_blank_lines) ... ok
test_validator_detects_duplicate_pairs (__main__.TestCorpusPreprocessing.test_validator_detects_duplicate_pairs) ... ok
test_wrapper_rejects_raw_line_count_mismatch (__main__.TestCorpusPreprocessing.test_wrapper_rejects_raw_line_count_mismatch) ... ok
test_bleu_metric_is_available (__main__.TestEnvironmentSetupAndReproducibility.test_bleu_metric_is_available) ... ok
test_expected_assignment_sizes_are_respected (__main__.TestEnvironmentSetupAndReproducibility.test_expected_assignment_sizes_are_respected) ... ok
test_output_directories_exist (__main__.TestEnvironmentSetupAndReproduci

## SECTION 9

## 9. Environment setup and reproducibility

This section prepares the Google Colab environment for the English↔Spanish iterative back-translation experiment. It mounts Google Drive, loads all required dependencies, defines the fixed paths and runtime settings, establishes reproducibility-oriented configuration, and validates that the notebook is ready to run from start to finish without manual fixes.

In [20]:
# =========================
# CELL 13: INSTALL REQUIRED LIBRARIES FOR THE FINAL NOTEBOOK
# =========================

%%capture
!pip install -q "transformers[torch,sentencepiece]>=4.38,<5.0"
!pip install -q "datasets>=2.18,<4.0"
!pip install -q "evaluate>=0.4,<0.5"
!pip install -q "sacremoses>=0.1,<0.2"
!pip install -q "sacrebleu>=2.4,<3.0"
!pip install -q "accelerate>=0.27,<1.0"

In [21]:
# =========================
# CELL 14: ENVIRONMENT CONFIGURATION AND REPRODUCIBILITY
# =========================

from google.colab import drive
import os
import random
import platform
from pathlib import Path

import numpy as np
import torch

# Mount Google Drive so that all corpora stored in /content/drive/MyDrive/anlp/ are accessible
drive.mount("/content/drive", force_remount=True)

# Avoid excessive tokenizer warnings / thread contention in Colab
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONHASHSEED"] = "42"

def set_global_reproducibility(random_seed: int) -> None:
    """
    Configure the main sources of randomness so that the notebook behaves
    as reproducibly as possible across runs in Colab.
    """
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(random_seed)
        torch.cuda.manual_seed_all(random_seed)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

random_seed = 42
set_global_reproducibility(random_seed)

# ==========================================================
# FIXED ROOT DIRECTORY REQUIRED BY THE ASSIGNMENT
# ==========================================================
corpus_root_directory = Path("/content/drive/MyDrive/anlp")
corpus_root_directory.mkdir(parents=True, exist_ok=True)

# ==========================================================
# LANGUAGE PAIR AND SELECTED PRETRAINED MODELS
# ==========================================================
source_language_code = "en"
target_language_code = "es"

forward_translation_model_name = "Helsinki-NLP/opus-mt-en-es"
backward_translation_model_name = "Helsinki-NLP/opus-mt-es-en"

# ==========================================================
# FIXED INPUT FILES CREATED IN THE PREVIOUS SECTIONS
# ==========================================================
parallel_training_source_path = corpus_root_directory / "news_commentary_train.en"
parallel_training_target_path = corpus_root_directory / "news_commentary_train.es"

development_source_path = corpus_root_directory / "news_commentary_dev.en"
development_target_path = corpus_root_directory / "news_commentary_dev.es"

test_source_path = corpus_root_directory / "news_commentary_test.en"
test_target_path = corpus_root_directory / "news_commentary_test.es"

english_monolingual_path = corpus_root_directory / "news_commentary_mono.en"
spanish_monolingual_path = corpus_root_directory / "news_commentary_mono.es"

# ==========================================================
# OUTPUT DIRECTORIES FOR THE TWO DIRECTIONAL MODELS
# ==========================================================
forward_model_output_directory = corpus_root_directory / "iterative_backtranslation_en_to_es"
backward_model_output_directory = corpus_root_directory / "iterative_backtranslation_es_to_en"

forward_model_output_directory.mkdir(parents=True, exist_ok=True)
backward_model_output_directory.mkdir(parents=True, exist_ok=True)

# ==========================================================
# FIXED DATA SIZES REQUIRED BY THE ASSIGNMENT
# ==========================================================
parallel_training_size = 1000
development_parallel_size = 200
test_parallel_size = 200
english_monolingual_size = 1000
spanish_monolingual_size = 1000

# ==========================================================
# TRAINING / GENERATION / EVALUATION SETTINGS
# ==========================================================
num_backtranslation_iterations = 3
early_stopping_patience = 3

max_source_sequence_length = 128
max_target_sequence_length = 128

learning_rate = 2e-5
training_batch_size = 16 if torch.cuda.is_available() else 8
evaluation_batch_size = training_batch_size

execution_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipeline_device = 0 if torch.cuda.is_available() else -1
use_fp16_training = torch.cuda.is_available()

print("Environment configuration completed successfully.\n")
print(f"Python version               : {platform.python_version()}")
print(f"PyTorch version              : {torch.__version__}")
print(f"Execution device             : {execution_device}")
print(f"Pipeline device              : {pipeline_device}")
print(f"Mixed precision enabled      : {use_fp16_training}")
print(f"Corpus root directory        : {corpus_root_directory}")
print(f"Forward model                : {forward_translation_model_name}")
print(f"Backward model               : {backward_translation_model_name}")
print(f"Forward model output folder  : {forward_model_output_directory}")
print(f"Backward model output folder : {backward_model_output_directory}")

Mounted at /content/drive
Environment configuration completed successfully.

Python version               : 3.12.12
PyTorch version              : 2.10.0+cu128
Execution device             : cuda
Pipeline device              : 0
Mixed precision enabled      : True
Corpus root directory        : /content/drive/MyDrive/anlp
Forward model                : Helsinki-NLP/opus-mt-en-es
Backward model               : Helsinki-NLP/opus-mt-es-en
Forward model output folder  : /content/drive/MyDrive/anlp/iterative_backtranslation_en_to_es
Backward model output folder : /content/drive/MyDrive/anlp/iterative_backtranslation_es_to_en


In [22]:
# =========================
# CELL 15: VALIDATE FILES, PATHS, AND RUNTIME ASSUMPTIONS
# =========================

from importlib.metadata import version

def count_lines_in_text_file(file_path: Path) -> int:
    with file_path.open("r", encoding="utf-8") as input_file:
        return sum(1 for _ in input_file)

def validate_environment_setup() -> dict:
    required_input_files = {
        "parallel_training_source": parallel_training_source_path,
        "parallel_training_target": parallel_training_target_path,
        "development_source": development_source_path,
        "development_target": development_target_path,
        "test_source": test_source_path,
        "test_target": test_target_path,
        "english_monolingual": english_monolingual_path,
        "spanish_monolingual": spanish_monolingual_path,
    }

    for file_description, file_path in required_input_files.items():
        if not file_path.exists():
            raise FileNotFoundError(
                f"Missing required file for Section 9: {file_description} -> {file_path}"
            )

    line_count_summary = {
        "parallel_training_source_lines": count_lines_in_text_file(parallel_training_source_path),
        "parallel_training_target_lines": count_lines_in_text_file(parallel_training_target_path),
        "development_source_lines": count_lines_in_text_file(development_source_path),
        "development_target_lines": count_lines_in_text_file(development_target_path),
        "test_source_lines": count_lines_in_text_file(test_source_path),
        "test_target_lines": count_lines_in_text_file(test_target_path),
        "english_monolingual_lines": count_lines_in_text_file(english_monolingual_path),
        "spanish_monolingual_lines": count_lines_in_text_file(spanish_monolingual_path),
    }

    assert line_count_summary["parallel_training_source_lines"] == parallel_training_size, \
        "The English training file does not contain the required 1000 lines."
    assert line_count_summary["parallel_training_target_lines"] == parallel_training_size, \
        "The Spanish training file does not contain the required 1000 lines."

    assert line_count_summary["development_source_lines"] == development_parallel_size, \
        "The English development file does not contain the required 200 lines."
    assert line_count_summary["development_target_lines"] == development_parallel_size, \
        "The Spanish development file does not contain the required 200 lines."

    assert line_count_summary["test_source_lines"] == test_parallel_size, \
        "The English test file does not contain the required 200 lines."
    assert line_count_summary["test_target_lines"] == test_parallel_size, \
        "The Spanish test file does not contain the required 200 lines."

    assert line_count_summary["english_monolingual_lines"] == english_monolingual_size, \
        "The English monolingual file does not contain the required 1000 lines."
    assert line_count_summary["spanish_monolingual_lines"] == spanish_monolingual_size, \
        "The Spanish monolingual file does not contain the required 1000 lines."

    assert line_count_summary["parallel_training_source_lines"] == line_count_summary["parallel_training_target_lines"], \
        "Training Moses files are misaligned."
    assert line_count_summary["development_source_lines"] == line_count_summary["development_target_lines"], \
        "Development Moses files are misaligned."
    assert line_count_summary["test_source_lines"] == line_count_summary["test_target_lines"], \
        "Test Moses files are misaligned."

    return {
        "required_files_exist": True,
        "expected_line_counts_match_assignment": True,
        "training_moses_alignment_ok": True,
        "development_moses_alignment_ok": True,
        "test_moses_alignment_ok": True,
        "forward_output_directory_exists": forward_model_output_directory.exists(),
        "backward_output_directory_exists": backward_model_output_directory.exists(),
        **line_count_summary,
    }

environment_validation_summary = validate_environment_setup()

print("Installed package versions:")
print(f" - transformers : {version('transformers')}")
print(f" - datasets     : {version('datasets')}")
print(f" - evaluate     : {version('evaluate')}")
print(f" - sacrebleu    : {version('sacrebleu')}")
print(f" - sacremoses   : {version('sacremoses')}")
print(f" - accelerate   : {version('accelerate')}")

print("\nEnvironment validation summary:")
for summary_key, summary_value in environment_validation_summary.items():
    print(f"{summary_key}: {summary_value}")

if not torch.cuda.is_available():
    print("\nWarning: GPU is not available.")
    print("The notebook can still run logically, but fine-tuning will be much slower.")
else:
    print("\nGPU detected correctly. The notebook is ready for seq2seq fine-tuning.")

Installed package versions:
 - transformers : 4.57.6
 - datasets     : 3.6.0
 - evaluate     : 0.4.6
 - sacrebleu    : 2.6.0
 - sacremoses   : 0.1.1
 - accelerate   : 0.34.2

Environment validation summary:
required_files_exist: True
expected_line_counts_match_assignment: True
training_moses_alignment_ok: True
development_moses_alignment_ok: True
test_moses_alignment_ok: True
forward_output_directory_exists: True
backward_output_directory_exists: True
parallel_training_source_lines: 1000
parallel_training_target_lines: 1000
development_source_lines: 200
development_target_lines: 200
test_source_lines: 200
test_target_lines: 200
english_monolingual_lines: 1000
spanish_monolingual_lines: 1000

GPU detected correctly. The notebook is ready for seq2seq fine-tuning.


In [23]:
# =========================
# CELL 16: LOAD THE MAIN DEPENDENCIES AND RUN A SMOKE TEST
# =========================

from datasets import Dataset
import evaluate

from transformers import (
    AutoConfig,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    pipeline,
)

bleu_metric = evaluate.load("sacrebleu")

forward_tokenizer = AutoTokenizer.from_pretrained(forward_translation_model_name)
backward_tokenizer = AutoTokenizer.from_pretrained(backward_translation_model_name)

forward_model_configuration = AutoConfig.from_pretrained(forward_translation_model_name)
backward_model_configuration = AutoConfig.from_pretrained(backward_translation_model_name)

def load_seq2seq_model_for_training(pretrained_model_name: str) -> AutoModelForSeq2SeqLM:
    """
    Load a Helsinki-NLP seq2seq model and place it on the configured device.
    This helper will be reused in the later training sections.
    """
    loaded_model = AutoModelForSeq2SeqLM.from_pretrained(pretrained_model_name)
    return loaded_model.to(execution_device)

def build_translation_pipeline(pretrained_model_name: str):
    """
    Create a translation pipeline for inference or synthetic-data generation.
    """
    return pipeline(
        task="translation",
        model=pretrained_model_name,
        tokenizer=pretrained_model_name,
        device=pipeline_device,
        batch_size=evaluation_batch_size,
    )

forward_tokenized_example = forward_tokenizer(
    "This is a short environment smoke test.",
    return_tensors="pt"
)
backward_tokenized_example = backward_tokenizer(
    "Esta es una prueba corta del entorno.",
    return_tensors="pt"
)

print("Main dependencies loaded successfully.")
print(f"Forward tokenizer vocabulary size  : {forward_tokenizer.vocab_size}")
print(f"Backward tokenizer vocabulary size : {backward_tokenizer.vocab_size}")
print(f"Forward sample token count         : {forward_tokenized_example['input_ids'].shape[1]}")
print(f"Backward sample token count        : {backward_tokenized_example['input_ids'].shape[1]}")
print("BLEU metric loaded correctly.")
print("The notebook environment is ready for dataset loading, fine-tuning, generation, and evaluation.")

Main dependencies loaded successfully.
Forward tokenizer vocabulary size  : 65001
Backward tokenizer vocabulary size : 65001
Forward sample token count         : 9
Backward sample token count        : 9
BLEU metric loaded correctly.
The notebook environment is ready for dataset loading, fine-tuning, generation, and evaluation.


In [25]:
# =========================
# CELL 17: UNIT TESTS FOR ENVIRONMENT SETUP AND REPRODUCIBILITY
# =========================

import unittest

class TestEnvironmentSetupAndReproducibility(unittest.TestCase):

    def test_required_corpus_files_exist(self):
        required_files = [
            parallel_training_source_path,
            parallel_training_target_path,
            development_source_path,
            development_target_path,
            test_source_path,
            test_target_path,
            english_monolingual_path,
            spanish_monolingual_path,
        ]

        for file_path in required_files:
            self.assertTrue(file_path.exists(), f"Missing required file: {file_path}")

    def test_expected_assignment_sizes_are_respected(self):
        self.assertEqual(count_lines_in_text_file(parallel_training_source_path), parallel_training_size)
        self.assertEqual(count_lines_in_text_file(parallel_training_target_path), parallel_training_size)

        self.assertEqual(count_lines_in_text_file(development_source_path), development_parallel_size)
        self.assertEqual(count_lines_in_text_file(development_target_path), development_parallel_size)

        self.assertEqual(count_lines_in_text_file(test_source_path), test_parallel_size)
        self.assertEqual(count_lines_in_text_file(test_target_path), test_parallel_size)

        self.assertEqual(count_lines_in_text_file(english_monolingual_path), english_monolingual_size)
        self.assertEqual(count_lines_in_text_file(spanish_monolingual_path), spanish_monolingual_size)

    def test_parallel_files_are_line_aligned(self):
        self.assertEqual(
            count_lines_in_text_file(parallel_training_source_path),
            count_lines_in_text_file(parallel_training_target_path)
        )
        self.assertEqual(
            count_lines_in_text_file(development_source_path),
            count_lines_in_text_file(development_target_path)
        )
        self.assertEqual(
            count_lines_in_text_file(test_source_path),
            count_lines_in_text_file(test_target_path)
        )

    def test_selected_model_names_match_the_chosen_directions(self):
        self.assertIn("en-es", forward_translation_model_name)
        self.assertIn("es-en", backward_translation_model_name)

    def test_output_directories_exist(self):
        self.assertTrue(forward_model_output_directory.exists())
        self.assertTrue(backward_model_output_directory.exists())

    def test_bleu_metric_is_available(self):
        reference_sentence = "this is a simple test sentence for bleu evaluation"
        bleu_result = bleu_metric.compute(
            predictions=[reference_sentence],
            references=[[reference_sentence]]
        )
        self.assertIn("score", bleu_result)
        self.assertGreaterEqual(bleu_result["score"], 99.0)

    def test_seed_reset_is_deterministic(self):
        set_global_reproducibility(random_seed)
        python_random_first = random.random()
        numpy_random_first = np.random.rand()
        torch_random_first = torch.rand(1).item()

        set_global_reproducibility(random_seed)
        python_random_second = random.random()
        numpy_random_second = np.random.rand()
        torch_random_second = torch.rand(1).item()

        self.assertAlmostEqual(python_random_first, python_random_second, places=12)
        self.assertAlmostEqual(numpy_random_first, numpy_random_second, places=12)
        self.assertAlmostEqual(torch_random_first, torch_random_second, places=12)

    def test_tokenizers_can_process_text(self):
        forward_ids = forward_tokenizer("Environment test sentence.")["input_ids"]
        backward_ids = backward_tokenizer("Frase de prueba del entorno.")["input_ids"]

        self.assertTrue(len(forward_ids) > 0)
        self.assertTrue(len(backward_ids) > 0)

    def test_runtime_configuration_is_consistent(self):
        self.assertEqual(use_fp16_training, torch.cuda.is_available())
        self.assertIn(pipeline_device, [0, -1])
        self.assertIn(source_language_code, ["en"])
        self.assertIn(target_language_code, ["es"])

unittest.main(argv=[""], verbosity=2, exit=False)

test_overwrite_true_allows_rerun (__main__.TestCorpusPreprocessing.test_overwrite_true_allows_rerun) ... ok
test_successful_cleaning_removes_duplicates_empties_and_normalizes_spacing (__main__.TestCorpusPreprocessing.test_successful_cleaning_removes_duplicates_empties_and_normalizes_spacing) ... ok
test_validator_detects_blank_lines (__main__.TestCorpusPreprocessing.test_validator_detects_blank_lines) ... ok
test_validator_detects_duplicate_pairs (__main__.TestCorpusPreprocessing.test_validator_detects_duplicate_pairs) ... ok
test_wrapper_rejects_raw_line_count_mismatch (__main__.TestCorpusPreprocessing.test_wrapper_rejects_raw_line_count_mismatch) ... ok
test_bleu_metric_is_available (__main__.TestEnvironmentSetupAndReproducibility.test_bleu_metric_is_available) ... ok
test_expected_assignment_sizes_are_respected (__main__.TestEnvironmentSetupAndReproducibility.test_expected_assignment_sizes_are_respected) ... ok
test_output_directories_exist (__main__.TestEnvironmentSetupAndReproduci

## Translate a sentence with the selected model

* The text to be translated must be assigned to the variable `text`

## Generate a dataset from a parallel corpus in moses format
The moses format consists of two separate files with the same amount of lines so that the segments in the *n*-th line in both documents are mutual translation. The parallel corpus must not contain blank or duplicated entries.

Once created, the dataset is split into training, development and testing.

In [26]:
# =========================
# REPLACEMENT CELL: LOAD THE PREDEFINED EXPERIMENTAL SPLITS
# =========================

from datasets import Dataset

def build_parallel_translation_dataset(source_file_path, target_file_path, source_language_code, target_language_code):
    source_sentences = read_lines(source_file_path)
    target_sentences = read_lines(target_file_path)

    if len(source_sentences) != len(target_sentences):
        raise ValueError(
            f"Parallel files do not have the same number of lines: "
            f"{source_file_path} has {len(source_sentences)}, "
            f"{target_file_path} has {len(target_sentences)}."
        )

    return Dataset.from_dict({
        "translation": [
            {
                source_language_code: source_sentence,
                target_language_code: target_sentence
            }
            for source_sentence, target_sentence in zip(source_sentences, target_sentences)
        ]
    })

train_dataset = build_parallel_translation_dataset(
    parallel_training_source_path,
    parallel_training_target_path,
    source,
    target,
)

dev_dataset = build_parallel_translation_dataset(
    development_source_path,
    development_target_path,
    source,
    target,
)

test_dataset = build_parallel_translation_dataset(
    test_source_path,
    test_target_path,
    source,
    target,
)

english_monolingual_sentences = read_lines(english_monolingual_path)
spanish_monolingual_sentences = read_lines(spanish_monolingual_path)

english_monolingual_dataset = Dataset.from_dict({"text": english_monolingual_sentences})
spanish_monolingual_dataset = Dataset.from_dict({"text": spanish_monolingual_sentences})

print("Final dataset sizes loaded from the non-overlapping split:")
print("Parallel training size :", len(train_dataset))
print("Development size       :", len(dev_dataset))
print("Test size              :", len(test_dataset))
print("English monolingual    :", len(english_monolingual_dataset))
print("Spanish monolingual    :", len(spanish_monolingual_dataset))

print("\nExample parallel training pair:")
print(train_dataset[0])

print("\nExample English monolingual sentence:")
print(english_monolingual_dataset[0])

print("\nExample Spanish monolingual sentence:")
print(spanish_monolingual_dataset[0])

Final dataset sizes loaded from the non-overlapping split:
Parallel training size : 1000
Development size       : 200
Test size              : 200
English monolingual    : 1000
Spanish monolingual    : 1000

Example parallel training pair:
{'translation': {'en': 'Arguably, Russia has improved its relations with China under Putin, but at the cost of acceding to China’s demands for two big disputed islands over which the two countries fought in 1969.', 'es': 'Supuestamente bajo Putin Rusia ha mejorado sus relaciones con China, pero al coste de ceder a las demandas chinas de dos grandes islas en disputa, por las cuales ambos países entraron en conflicto en 1969.'}}

Example English monolingual sentence:
{'text': 'But it is as if, owing to an excess of prudence, they cannot bring themselves to pursue their own arguments to their logical conclusions.'}

Example Spanish monolingual sentence:
{'text': 'Pero es como si, debido a un exceso de prudencia, no pudieran llevar sus propios argumentos

## Translate the test set using the pre-trained model



In [ ]:
from transformers import pipeline
import evaluate

# Get the sentences in the test set
inputs = [ex[source] for ex in test_dataset["translation"]]
references = [ex[target] for ex in test_dataset["translation"]]

# Translate using pipelines - Use GPU 0 (device="cuda:0")
translator = pipeline("translation", model=model_name, device="cuda:0", batch_size=64)
pre_outputs = translator(inputs)
outputs = [ex["translation_text"] for ex in pre_outputs]

metric = evaluate.load("sacrebleu") # BLEU
result = metric.compute(predictions=outputs, references=references)
print (result)

del translator

## Preprocess the datasets before their use for fine tuning
The proprocessing implies tokenizing the sentences in the datasets using the tokenizer included in the pre-trained model


In [ ]:
from transformers import AutoTokenizer

max_input_length = 128
max_target_length = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
    inputs = [ex[source] for ex in examples["translation"]]
    targets = [ex[target] for ex in examples["translation"]]
    model_inputs = tokenizer(text=inputs, max_length=max_input_length, padding=True, truncation=True)
    labels = tokenizer(text_target=targets, max_length=max_target_length, padding=True, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True)
tokenized_dev_dataset = dev_dataset.map(preprocess_function, batched=True)

## Fine tune the model on the dataset created above
Before fine tuning, we need to set the automatic evaluation metric to be used to evaluate on the development set, then we will run the training algorithm on the training dataset


### Define the metric to be used on the development set

In [ ]:
import evaluate

metric = evaluate.load("sacrebleu") # BLEU

import numpy as np
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

### Fine tune the model

In [ ]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda:0") # Load in GPU 0

args = Seq2SeqTrainingArguments(
    output_dir="./"+output_model_name,
    eval_strategy="epoch",
    save_strategy="epoch",
    #evaluation_strategy="steps",
    #save_strategy="steps",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=30,
    predict_with_generate=True,
    fp16=True,
    metric_for_best_model="bleu",
    load_best_model_at_end=True, # It uses metric_for_best_model to compare models
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_dev_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)], # It uses metric_for_best_model
)

trainer.train()
trainer.save_model()

del trainer
del model
del data_collator

## Translate the test set using the fine-tuned model

In [ ]:
from transformers import pipeline
import evaluate

# Get the sentences in the test set
inputs = [ex[source] for ex in test_dataset["translation"]]
references = [ex[target] for ex in test_dataset["translation"]]

# Translate using pipelines - Use GPU 0 (device="cuda:0")
translator = pipeline("translation", model=output_model_name, device="cuda:0", batch_size=64)
pre_outputs = translator(inputs)
outputs = [ex["translation_text"] for ex in pre_outputs]

metric = evaluate.load("sacrebleu") # BLEU
result = metric.compute(predictions=outputs, references=references)
print (result)

del translator